In [ ]:
from Libraries.inference_training import Configuration, ImageDataset
from Libraries.inference_training import initCudaEnvironment, createTransforms
from Libraries.inference_training import drawImageAndFeatureMasks
from Libraries.inference_training import exportOnnxModel, writeONNXMeta, loadONNX
from Libraries.inference_training import trainModel, saveModel, loadModel
from Libraries.inference_training import createModelInstance, testInference
import os
import random
from paths import *

In [ ]:
initCudaEnvironment(numCudaDevices=1,
                    visibleCudaDevices="0",
                    clearCudaDeviceCount=False)

# model create function

In [ ]:
def createModel(trainDirectory:str, testDirectory:str, modelName:str, epochs:int, labels: list[str], augment_data:bool, model_path:str, model_description="default"):
    """
    :param trainDirectory: path naar training dataset
    :param testDirectory: path naar testing dataset
    :param modelName: naam van model 
    :param epochs: hoeveelheid epochs
    :param labels: list van labels, geef normaal ["parkeerplaatsen"] als er geen andere objecten zijn
    :param augment_data: bepaald of er image transforms gedaan worden, nog niet getest
    :param model_path: path naar save locatie van model
    :param model_description: beschrijft het model in de ONNX als het gesaved is
    :return: 
    """
    config = Configuration()
    print("Device: " + str(config.device))
    
    config.setDatasetPaths(trainPath=trainDirectory, testPath=testDirectory)
    config.setFilePrefix("")
    config.setModelName(modelName)
    config.setInputSizes(inputWidth=250, inputHeight=250)
    config.setInputCellSize(cellSizeM=0.25, minCellSizeM=0.1, maxCellSizeM=0.5)
    config.setVersion(20250121)
    config.setModelInfo(channels=3, numClasses=2+1,  # (1 + background)
                        bboxOverlap=True, bboxPerImage=250, reuseModel=False)
    config.setEpochs(epochs)
    config.setOnnxInfo(producer="Tygron", description=model_description)
    config.addLegendEntry("Background", 0, "#00000000")
    config.setSavePath(model_path)
    i = 1
    for label in labels:
        config.addLegendEntry(label, i, ["#"+''.join([random.choice('ABCDEF0123456789') for i in range(6)])])
        i += 1
    
    config.setOnnxMetaData(scoreThreshold=0.2,
                           maskThreshold=0.3,
                           strideFraction=0.5)
    
    config.setTensorInfo(tensorName='input_A:RGB_normalized', batchAmount=1)
    if augment_data:
        trainingDataset = ImageDataset(config, True, createTransforms(True))
        testDataset = ImageDataset(config, False, createTransforms(False))
    else:
        trainingDataset = ImageDataset(config, True, createTransforms(False))
        testDataset = ImageDataset(config, False, createTransforms(False))
    
    print("Train Image count: "+str(trainingDataset.__len__()))
    print("Test Image count: "+str(testDataset.__len__()))
    
    if not trainingDataset.validateFiles(False):
        print("Inconsistent training dataset ")
        trainingDataset.validateFiles(True)
    
    if not testDataset.validateFiles(False):
        print("Inconsistent test dataset ")
        testDataset.validateFiles(True)
    
    print("Pytorch model name " + config.getPytorchModelFileName())
    print("Onnx file name " + config.getOnnxFileName())
    
    model = trainModel(config, trainingDataset, testDataset)
    model.eval()
    
    saveModel(config, model, path=model_path+config.getPytorchModelFileName())
    
    exportOnnxModel(config, model, True)
    writeONNXMeta(config)
    
    return model, config

# load existing model

In [ ]:
def load_model(path:str, config=None):
    """
    :param path: path naar model save location 
    :param config: config als het eerder is ingesteld
    :return: 
    """
    if not config:
        config = Configuration()
    
    model = createModelInstance(config)
    loadModel(config, model, path)
    
    return model, config

In [ ]:
def load_default_config():
    config = Configuration()
    config.setFilePrefix("")
    config.setInputSizes(inputWidth=250, inputHeight=250)
    config.setInputCellSize(cellSizeM=0.25, minCellSizeM=0.1, maxCellSizeM=0.5)
    config.setVersion(20250121)
    config.setModelInfo(channels=3, numClasses=2+1,  # (1 + background)
                        bboxOverlap=True, bboxPerImage=250, reuseModel=False)
    config.addLegendEntry("Background", 0, "#00000000")
    return config

# combo data model

In [ ]:
modelName = "combo_sets_model"
epochs = 25
labels = ["parkeerplaats"]
augment = False
combo_model, config = createModel(COMBO_TRAIN, COMBO_TEST, modelName, epochs, labels, augment, MODELS_DIR)

# result testing

In [ ]:
config1 = load_default_config()
model, config1 = load_model(MODELS_DIR / "combo_sets_model.pt",config1)

In [ ]:
model.eval()
config1.setDatasetPaths(trainPath=COMBO_TRAIN,testPath=COMBO_TEST)
testDataset = ImageDataset(config1, False, createTransforms(False))

In [ ]:
testDataset.validateFiles(False)
config1.setOnnxMetaData(scoreThreshold=0.2,
                        maskThreshold=0.3,
                        strideFraction=0.5)

In [ ]:
import torch
from torchvision import tv_tensors
from torchvision.transforms import v2

def compute_dice_score(pred_masks, gt_masks):
    """Compute Dice score between predicted and ground truth masks."""
    dice_scores = []
    for pred_mask, gt_mask in zip(pred_masks, gt_masks):
        intersection = (pred_mask & gt_mask).sum().item()
        union = pred_mask.sum().item() + gt_mask.sum().item()
        if union == 0:
            dice_scores.append(1.0 if intersection == 0 else 0.0)  # Handle edge cases
        else:
            dice_scores.append(2 * intersection / union)
    if len(dice_scores) == 0:
        return None
    return sum(dice_scores) / len(dice_scores)  # Mean Dice score

def testInferenceDice(config: Configuration,
                      dataset: ImageDataset, model,
                      imageNumber: int):
    imgs = dataset.getImages(imageNumber)
    for i in range(len(imgs)):
        imgs[i] = v2.functional.convert_image_dtype(imgs[i], dtype=torch.float)
        imgs[i] = tv_tensors.Image(imgs[i])

    img = torch.cat(imgs, 0) 
    eval_transform = createTransforms(train=False)

    with torch.no_grad():
        x = eval_transform(img)
        x = x.to(config.device)
        predictions = model([x])
        pred = predictions[0]

    pred_masks = (pred["masks"] > config.maskThreshold).squeeze(1).int()
    pred_masks = pred_masks.to(config.device)

    # Get ground truth masks
    gt_masks = dataset.getMask(imageNumber)  # Adjust this to match your dataset structure
    gt_masks = (gt_masks > 0).int()  # Convert to binary masks
    gt_masks = gt_masks.to(config.device)
    
    # Ensure pred_masks and gt_masks have the same shape
    if pred_masks.shape != gt_masks.shape:
        gt_masks = torch.nn.functional.interpolate(
            gt_masks.unsqueeze(0).float(), size=pred_masks.shape[-2:][0], mode="nearest"
        ).squeeze(0).int()
    print(pred_masks.shape, gt_masks.shape)
    print("pred masks:",pred_masks)
    print("gt masks:",gt_masks)
        # Compute Dice score
    dice_score = compute_dice_score(pred_masks, gt_masks)

    return dice_score


In [ ]:
import numpy as np
length = testDataset.__len__()
print(length)
scores = []
for i in range(length):
    scores.append(testInferenceDice(config1, testDataset, model, i))

In [ ]:
i = 0
j = 0
for k in scores:
    if isinstance(k, float):
        i += k
        j+= 1

print(i/j)